<a href="https://colab.research.google.com/github/magdoch/ML_python_projects/blob/main/%D0%92%D0%B8%D0%BA%D0%BE%D1%80%D0%B8%D1%81%D1%82%D0%B0%D0%BD%D0%BD%D1%8F_%D0%BF%D1%80%D0%BE%D0%BC%D0%BF%D1%82%D1%96%D0%B2_%D1%96_%D0%B0%D0%B3%D0%B5%D0%BD%D1%82%D1%96%D0%B2_%D0%B2_Langchain_Soroka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



### Завдання 1: Виклик LLM з базовим промптом

**Мета:** навчитися викликати LLM через LangChain зі звичайним текстовим промптом.

**Що потрібно зробити:**

1. Створіть промпт, який дозволяє отримати інформацію простою мовою на тему "Квантові обчислення". Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

2. Обмежте відповідь до 200 символів і пропишіть в промпті, аби відповідь була короткою (це зекономить вам час і гроші на згенеровані токени).

3. Встановіть своє значення температури на власний розсуд (тут немає правильного чи неправильного значення) і напишіть коментарем, чому ви обрали саме таке значення для цього завдання.

**Вибір моделі:** можна скористатись як моделлю з HuggingFace, так і ChatGPT будь-якої версії, яка вам до вподоби і пасує за прайсингом. В обох випадках потрібно імпортувати відповідний клас з LangChain для виклику LLM за API.

**Мова запитів:** промпти можна писати як українською, так і англійською — орієнтуйтесь на те, де і для чого ви хочете потім використовувати цей проєкт. У розв'язках промпти — українською.

---

**🔐 Як безпечно зберігати і підвантажувати API-ключі**

API-токен потрібно зчитувати з безпечного джерела, а **не хардкодити в ноутбуці**. Якщо хтось отримає доступ до вашого ключа, він буде витрачати токени за ваш рахунок, а вам це не треба :)

Є кілька способів. Перший ми використовували на лекції, ще два для розширення вашого розуміння, як ще це можна зробити і що шлях не лише один. Для виконання цього ДЗ можете використовувати будь-який спосіб підвантаження ключів у ноутбук.

**Спосіб 1: Файл `creds.json` (рекомендований)**

Створіть файл `creds.json` з вашими ключами, завантажте його в Google Colab під час роботи, але **не здавайте** цей файл у ДЗ і **не комітьте** в git.

```python
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["HF_TOKEN"]
```

**Спосіб 2: Google Colab Secrets**

У лівій панелі Colab натисніть іконку 🔑 (Secrets) → "Add new secret" → введіть назву (наприклад, `HF_TOKEN`) та значення ключа → увімкніть тогл доступу для ноутбука.

```python
from google.colab import userdata
api_key = userdata.get("HF_TOKEN")
```

Зручно тим, що ключ зберігається в акаунті і доступний у всіх ваших ноутбуках. Мінус — при кожній новій сесії потрібно перевірити, що доступ увімкнено.

**Спосіб 3: Google AI Studio (для Gemini API)**

Якщо працюєте з моделями Google Gemini, отримати безкоштовний API-ключ можна в [Google AI Studio](https://aistudio.google.com/app/apikey): увійдіть з Google-акаунтом → натисніть "Get API key" → "Create API key". Далі використовуйте ключ через будь-який із способів вище.



In [58]:
!pip -q install langchain langchain_openai langchain-community huggingface_hub openai langchain-huggingface

In [59]:
!pip show langchain

Name: langchain
Version: 1.2.15
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [90]:
import json
import os

with open('creds.json') as f:
    creds = json.load(f)

api_key = creds["HF_TOKEN"]
openai_key = creds["OPENAI_API_KEY"]
serp_key= creds["SERPAPI_API_KEY"]

os.environ["OPENAI_API_KEY"] = openai_key

In [69]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

In [70]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    api_key=creds["OPENAI_API_KEY"]
)

In [71]:
prompt = (
   "Поясни простими словами квантові обчислення. "
    "Дай визначення, переваги та дослідження. "
    "До 200 символів."
)

In [72]:
from IPython.display import Markdown
response = llm.invoke([HumanMessage(content=prompt)])
Markdown(response.content)

Квантові обчислення — це обробка інформації за допомогою квантових бітів (кубітів), які можуть бути в кількох станах одночасно. Переваги: швидкість, ефективність у складних задачах. Дослідження тривають у багатьох сферах.

Температуру обрала низьку, щоб не було сильної імпровізації.

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [73]:
from langchain_core.prompts import PromptTemplate

In [78]:
prompt = PromptTemplate(
    input_variables=["thema"],
    template="""Поясни простими словами таку тему: {thema}.
    Структура:
    1. Визначення (1 речення).
    2. Головна перевага.
    3. Актуальність досліджень.
    Загальний обсяг: до 200 символів.""",
)

In [82]:
formatted_prompt = prompt.format(thema="Баєсівські методи в машинному навчанні")

response = llm.invoke(formatted_prompt)

print(response.content)

Баєсівські методи в машинному навчанні — це підходи, що ґрунтуються на теорії ймовірностей для прийняття рішень. Головна перевага — здатність працювати з невизначеністю. Вони актуальні для аналізу даних і прогнозування.


In [81]:
formatted_prompt = prompt.format(thema="Трансформери в машинному навчанні")


response = llm.invoke(formatted_prompt)

print(response.content)

Трансформери — це архітектура в машинному навчанні, що обробляє послідовності даних, наприклад, текст. Головна перевага — здатність ефективно враховувати контекст. Дослідження актуальні для покращення обробки мови та зображень.


In [83]:
formatted_prompt = prompt.format(thema="Explainable AI")


response = llm.invoke(formatted_prompt)

print(response.content)

Explainable AI (XAI) — це штучний інтелект, який може пояснити свої рішення та дії. Головна перевага — зрозумілість для користувачів. Дослідження актуальні для довіри та безпеки в AI.




### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [ ]:
!pip install -q langchain_community duckduckgo_search

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")

In [84]:
!pip install -q google-search-results langchain-community langchain_experimental

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 3.7 MB/s eta 0:00:00


In [85]:
!pip show langchain langchain-core langchain-community langchain-experimental langchain-classic langchainhub langsmith 2>/dev/null | grep -E "^(Name|Version)"

Name: langchain
Version: 1.2.15
Name: langchain-core
Version: 1.2.29
Name: langchain-community
Version: 0.4.1
Name: langchain-experimental
Version: 0.4.1
Name: langchain-classic
Version: 1.0.3
Name: langsmith
Version: 0.7.30


In [87]:
from langchain.agents import create_agent
from langchain_classic.agents import load_tools, Tool, AgentExecutor, AgentType, create_react_agent, initialize_agent
from langchain_classic import hub
from langchain_experimental.tools import PythonREPLTool

In [93]:
llm_agent = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
    api_key=creds["OPENAI_API_KEY"]
)

In [91]:
os.environ["SERPAPI_API_KEY"] = serp_key

In [92]:
tools = load_tools(["serpapi"], llm=llm)
prompt = hub.pull("hwchase17/react")

In [97]:
agent = create_react_agent(llm=llm_agent, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

result = agent_executor.invoke({
    "input": """Find the 5 most recent publications on artificial intelligence from 2026.
    List each one, including the title, authors and a brief description. Be sure to include the authors."""
})

print(result["output"])




> Entering new AgentExecutor chain...
I need to search for the most recent publications on artificial intelligence from 2026. Since I don't have access to future data, I will search for the latest publications available up to my last training cut-off in October 2023. 

Action: Search  
Action Input: "recent publications on artificial intelligence 2026"  ['The 2026 Stanford AI Index reveals how global AI trends 2026 are reshaping compute, emissions, and public trust in powerful models.', "1. Agentic AI isn't ready for prime time — yet · 2. The AI bubble will deflate, with economic ramifications · 3. Generative AI should become an ...", 'Top 7 Books on AI to Read in 2026 · 1. Empire of AI — Karen Hao · 2. AI Engineering — Chip Huyen · 3. Nexus — Yuval Noah Harari · 4. Co-Intelligence ...', 'Authors and titles for April 2026 ; [1] arXiv:2604.00005 · Moran Sun, Tianlin Li, Yuwei Zheng, Zhenhong Zhou, Aishan Liu, Xianglong Liu, Yang Liu ; [2] arXiv: ...', 'Download 2026 ReportBiotechnolog



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [104]:
tools = load_tools(["serpapi"])
react_prompt = hub.pull("hwchase17/react")

In [106]:
agent = create_react_agent(llm=llm_agent, tools=tools, prompt=react_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10
)

result = agent_executor.invoke({
    "input": """ You are a business analytics
    Historical data:
    - 2021: 200 tonnes
    - 2022: 190 tonnes
    - 2023: 210 tonnes
    - 2024 (year not yet over): 220 tonnes

    Estimate how many oranges we will be able to export in 2026, taking into account weather conditions in Brazil and global demand for oranges based on the economic situation.
    Give me a specific number of tonnes
    If you cannot find data, clearly state what is missing."""
})

Markdown(result["output"])



> Entering new AgentExecutor chain...
To estimate the number of oranges that can be exported in 2026, I need to consider historical data trends, potential weather conditions in Brazil, and the global demand for oranges influenced by the economic situation. 

The historical data shows a slight fluctuation in exports:
- 2021: 200 tonnes
- 2022: 190 tonnes (a decrease)
- 2023: 210 tonnes (an increase)
- 2024: 220 tonnes (projected increase)

From this data, there seems to be a general upward trend in exports from 2023 to 2024. However, the decrease in 2022 indicates that external factors, such as weather conditions or economic situations, can significantly impact exports.

To make a more accurate estimate for 2026, I need to gather information on:
1. Current weather conditions in Brazil and forecasts for the next few years.
2. The global economic situation and demand for oranges, particularly in key markets.

Since I don't have this information readily available, I will search for curre

231 tonnes